# 99 - Test Runner

Automated test orchestrator for F2.2, F3.2, F6.1 features.

**What it does:**
1. Runs the RDF parser with different input configurations
2. Runs schema detector and SHACL parser after each
3. Collects results and generates comparison report

**Prerequisites:**
- Environment `env_rdf_jena` attached
- Shortcuts to test data configured
- Notebooks synced to workspace

In [ ]:
# Test Configuration
from dataclasses import dataclass
from typing import List, Dict, Optional
import json
from datetime import datetime

# Define all test scenarios
@dataclass
class TestScenario:
    id: str
    name: str
    description: str
    input_paths: List[str]
    expected_level: int
    expected_min_triples: int
    checks: List[str]  # What to verify

# Test scenarios based on realistic user inputs
TEST_SCENARIOS = [
    # Format tests (F2.2)
    TestScenario(
        id="T1_TTL",
        name="Turtle Format (Informative)",
        description="Single combined TTL file with full schema",
        input_paths=["informative_nen2660/nen2660.ttl"],
        expected_level=4,
        expected_min_triples=5000,
        checks=["format_detection", "triple_count", "schema_level"]
    ),
    TestScenario(
        id="T2_RDFXML",
        name="RDF/XML Format",
        description="Same content in RDF/XML format",
        input_paths=["informative_nen2660/nen2660.rdf"],
        expected_level=4,
        expected_min_triples=5000,
        checks=["format_detection", "triple_count", "schema_level"]
    ),
    TestScenario(
        id="T3_JSONLD",
        name="JSON-LD Format",
        description="Same content in JSON-LD format",
        input_paths=["informative_nen2660/nen2660.jsonld"],
        expected_level=4,
        expected_min_triples=5000,
        checks=["format_detection", "triple_count", "schema_level"]
    ),
    TestScenario(
        id="T4_TRIG",
        name="TriG Format (Named Graphs)",
        description="Same content in TriG with named graphs",
        input_paths=["informative_nen2660/nen2660.trig"],
        expected_level=4,
        expected_min_triples=5000,
        checks=["format_detection", "triple_count", "schema_level", "named_graphs"]
    ),
    
    # Schema level tests (F3.2)
    TestScenario(
        id="T5_LEVEL0",
        name="Instance Data Only (Level 0)",
        description="Examples without schema - tests Level 0 detection",
        input_paths=["examples_nen2660"],
        expected_level=0,
        expected_min_triples=100,
        checks=["schema_level", "no_schema_predicates"]
    ),
    TestScenario(
        id="T6_LEVEL2",
        name="RDFS Schema Only (Level 2)",
        description="Only RDFS file - tests Level 2 detection",
        input_paths=["normative_nen2660/nen2660-rdfs.ttl"],
        expected_level=2,
        expected_min_triples=500,
        checks=["schema_level", "has_rdfs_class"]
    ),
    TestScenario(
        id="T7_LEVEL3",
        name="OWL Ontology (Level 3)",
        description="RDFS + OWL files - tests Level 3 detection",
        input_paths=["normative_nen2660/nen2660-rdfs.ttl", "normative_nen2660/nen2660-owl.ttl"],
        expected_level=3,
        expected_min_triples=1000,
        checks=["schema_level", "has_owl_class"]
    ),
    
    # SHACL test (F6.1)
    TestScenario(
        id="T8_SHACL",
        name="SHACL Shapes (Level 4)",
        description="SHACL file - tests shape parsing",
        input_paths=["normative_nen2660/nen2660-shacl.ttl"],
        expected_level=4,
        expected_min_triples=500,
        checks=["schema_level", "has_shacl_shapes", "shacl_constraints"]
    ),
    
    # Realistic user scenarios
    TestScenario(
        id="T9_FULL_NORMATIVE",
        name="Full Normative Schema",
        description="All 4 normative files - typical ontology import",
        input_paths=["normative_nen2660"],
        expected_level=4,
        expected_min_triples=5000,
        checks=["schema_level", "all_schema_layers"]
    ),
    TestScenario(
        id="T10_SCHEMA_PLUS_DATA",
        name="Schema + Instance Data",
        description="Full normative schema + example instances",
        input_paths=["normative_nen2660", "examples_nen2660"],
        expected_level=4,
        expected_min_triples=6000,
        checks=["schema_level", "has_instances", "full_pipeline"]
    ),
]

print(f"Defined {len(TEST_SCENARIOS)} test scenarios")
for ts in TEST_SCENARIOS:
    print(f"  {ts.id}: {ts.name}")

In [ ]:
@dataclass
class TestResult:
    scenario_id: str
    scenario_name: str
    status: str  # PASS, FAIL, ERROR
    triple_count: int
    schema_level: Optional[int]
    shacl_shape_count: int
    duration_seconds: float
    errors: List[str]
    details: Dict

# Results storage
test_results: List[TestResult] = []

In [ ]:
from pyspark.sql import functions as F
import time

def run_parser_for_scenario(scenario: TestScenario) -> int:
    """Run the RDF parser with specific input paths. Returns triple count."""
    # Set widget values for the parser notebook
    input_paths_str = ",".join(scenario.input_paths)
    
    # Use mssparkutils to run notebook with parameters
    try:
        mssparkutils.notebook.run(
            "01_rdf_parser_jena",
            timeout_seconds=600,
            arguments={
                "input_paths": input_paths_str,
                "output_table": "bronze_triples",
                "mode": "overwrite"
            }
        )
    except Exception as e:
        print(f"  Note: Notebook run completed or error: {e}")
    
    # Get triple count
    return spark.sql("SELECT COUNT(*) as cnt FROM bronze_triples").collect()[0].cnt


def run_schema_detector() -> Optional[int]:
    """Run schema detector and return detected level."""
    try:
        mssparkutils.notebook.run("02_schema_detector", timeout_seconds=300)
        # Read result from JSON
        with open("/lakehouse/default/Files/config/schema_statistics.json") as f:
            stats = json.load(f)
            return stats.get("schemaLevel")
    except Exception as e:
        print(f"  Schema detector error: {e}")
        return None


def run_shacl_parser() -> int:
    """Run SHACL parser and return shape count."""
    try:
        mssparkutils.notebook.run("10_shacl_parser", timeout_seconds=300)
        return spark.sql("SELECT COUNT(DISTINCT shape_uri) FROM silver_shacl_shapes").collect()[0][0]
    except Exception as e:
        print(f"  SHACL parser error: {e}")
        return 0

In [ ]:
def validate_scenario(scenario: TestScenario, triple_count: int, schema_level: int, shacl_count: int) -> List[str]:
    """Validate test results against expectations. Returns list of failures."""
    failures = []
    
    # Check triple count
    if triple_count < scenario.expected_min_triples:
        failures.append(f"Triple count {triple_count} < expected minimum {scenario.expected_min_triples}")
    
    # Check schema level
    if schema_level is not None and schema_level != scenario.expected_level:
        failures.append(f"Schema level {schema_level} != expected {scenario.expected_level}")
    
    # Check-specific validations
    if "has_shacl_shapes" in scenario.checks and shacl_count == 0:
        failures.append("Expected SHACL shapes but found none")
    
    return failures

In [ ]:
# Alternative: Direct execution without mssparkutils.notebook.run
# This approach reads and executes notebook code directly
# Useful if mssparkutils.notebook.run is not available

def run_scenario_direct(scenario: TestScenario) -> TestResult:
    """Run a test scenario by directly manipulating the data."""
    start_time = time.time()
    errors = []
    details = {}
    
    print(f"\n{'='*60}")
    print(f"Running: {scenario.id} - {scenario.name}")
    print(f"Input: {scenario.input_paths}")
    print(f"{'='*60}")
    
    try:
        # Step 1: Clear existing data
        spark.sql("DROP TABLE IF EXISTS bronze_triples")
        spark.sql("DROP TABLE IF EXISTS silver_shacl_shapes")
        print("Cleared existing tables")
        
        # Step 2: Run parser notebook with parameters
        print(f"\nRunning parser...")
        triple_count = run_parser_for_scenario(scenario)
        print(f"  Triples: {triple_count}")
        details["triple_count"] = triple_count
        
        # Step 3: Run schema detector
        print(f"\nRunning schema detector...")
        schema_level = run_schema_detector()
        print(f"  Schema level: {schema_level}")
        details["schema_level"] = schema_level
        
        # Step 4: Run SHACL parser if relevant
        shacl_count = 0
        if "shacl_constraints" in scenario.checks or scenario.expected_level == 4:
            print(f"\nRunning SHACL parser...")
            shacl_count = run_shacl_parser()
            print(f"  SHACL shapes: {shacl_count}")
        details["shacl_shapes"] = shacl_count
        
        # Step 5: Validate
        failures = validate_scenario(scenario, triple_count, schema_level, shacl_count)
        if failures:
            errors.extend(failures)
            status = "FAIL"
        else:
            status = "PASS"
            
    except Exception as e:
        errors.append(str(e))
        status = "ERROR"
        triple_count = 0
        schema_level = None
        shacl_count = 0
    
    duration = time.time() - start_time
    print(f"\nResult: {status} ({duration:.1f}s)")
    if errors:
        for err in errors:
            print(f"  - {err}")
    
    return TestResult(
        scenario_id=scenario.id,
        scenario_name=scenario.name,
        status=status,
        triple_count=triple_count,
        schema_level=schema_level,
        shacl_shape_count=shacl_count,
        duration_seconds=duration,
        errors=errors,
        details=details
    )

In [ ]:
# Select which tests to run (modify this list to run subset)
# Options: Run all, or specify test IDs

# Run ALL tests:
scenarios_to_run = TEST_SCENARIOS

# Or run specific tests:
# scenarios_to_run = [s for s in TEST_SCENARIOS if s.id in ["T1_TTL", "T2_RDFXML", "T3_JSONLD"]]

# Or run only format tests:
# scenarios_to_run = TEST_SCENARIOS[:4]

# Or run only schema level tests:
# scenarios_to_run = TEST_SCENARIOS[4:8]

print(f"Will run {len(scenarios_to_run)} test scenarios:")
for s in scenarios_to_run:
    print(f"  - {s.id}: {s.name}")

In [ ]:
# RUN ALL SELECTED TESTS
print("\n" + "#"*70)
print("# STARTING TEST EXECUTION")
print("#"*70)

total_start = time.time()
test_results = []

for i, scenario in enumerate(scenarios_to_run, 1):
    print(f"\n[{i}/{len(scenarios_to_run)}] {scenario.id}")
    result = run_scenario_direct(scenario)
    test_results.append(result)

total_duration = time.time() - total_start
print(f"\n\nTotal test time: {total_duration:.1f}s")

In [ ]:
# RESULTS SUMMARY
print("\n" + "="*70)
print("TEST RESULTS SUMMARY")
print("="*70)

passed = sum(1 for r in test_results if r.status == "PASS")
failed = sum(1 for r in test_results if r.status == "FAIL")
errors = sum(1 for r in test_results if r.status == "ERROR")

print(f"\nTotal: {len(test_results)} | Passed: {passed} | Failed: {failed} | Errors: {errors}")
print(f"Duration: {total_duration:.1f}s")

print("\n" + "-"*70)
print(f"{'Test ID':<15} {'Name':<30} {'Status':<8} {'Triples':<10} {'Level':<6} {'SHACL'}")
print("-"*70)

for r in test_results:
    level_str = str(r.schema_level) if r.schema_level is not None else "N/A"
    status_icon = "✅" if r.status == "PASS" else "❌" if r.status == "FAIL" else "⚠️"
    print(f"{r.scenario_id:<15} {r.scenario_name[:30]:<30} {status_icon} {r.status:<6} {r.triple_count:<10} {level_str:<6} {r.shacl_shape_count}")

# Show failures
if failed > 0 or errors > 0:
    print("\n" + "-"*70)
    print("FAILURES AND ERRORS:")
    print("-"*70)
    for r in test_results:
        if r.status != "PASS":
            print(f"\n{r.scenario_id}: {r.scenario_name}")
            for err in r.errors:
                print(f"  - {err}")

In [ ]:
# FORMAT CONSISTENCY CHECK
# Compare triple counts across formats (T1-T4 should have similar counts)

print("\n" + "="*70)
print("FORMAT CONSISTENCY CHECK (F2.2)")
print("="*70)

format_tests = [r for r in test_results if r.scenario_id.startswith("T") and int(r.scenario_id[1]) <= 4]

if len(format_tests) >= 2:
    counts = [r.triple_count for r in format_tests]
    avg_count = sum(counts) / len(counts)
    max_deviation = max(abs(c - avg_count) / avg_count * 100 for c in counts) if avg_count > 0 else 0
    
    print(f"\nTriple counts by format:")
    for r in format_tests:
        deviation = abs(r.triple_count - avg_count) / avg_count * 100 if avg_count > 0 else 0
        print(f"  {r.scenario_id}: {r.triple_count} triples ({deviation:.1f}% from mean)")
    
    print(f"\nAverage: {avg_count:.0f} triples")
    print(f"Max deviation: {max_deviation:.1f}%")
    
    if max_deviation < 5:
        print("\n✅ Format consistency: GOOD (< 5% deviation)")
    elif max_deviation < 10:
        print("\n⚠️ Format consistency: ACCEPTABLE (5-10% deviation)")
    else:
        print("\n❌ Format consistency: POOR (> 10% deviation) - investigate differences")
else:
    print("Not enough format tests to compare")

In [ ]:
# SCHEMA LEVEL DETECTION CHECK (F3.2)
print("\n" + "="*70)
print("SCHEMA LEVEL DETECTION CHECK (F3.2)")
print("="*70)

level_tests = [(r, s) for r, s in zip(test_results, scenarios_to_run) 
               if "LEVEL" in r.scenario_id or r.scenario_id in ["T1_TTL", "T8_SHACL"]]

print(f"\n{'Test':<15} {'Expected':<10} {'Actual':<10} {'Status'}")
print("-"*50)

level_issues = []
for result, scenario in level_tests:
    expected = scenario.expected_level
    actual = result.schema_level
    match = actual == expected if actual is not None else False
    status = "✅" if match else "❌"
    print(f"{result.scenario_id:<15} Level {expected:<7} Level {str(actual):<7} {status}")
    if not match:
        level_issues.append((result.scenario_id, expected, actual))

if not level_issues:
    print("\n✅ All schema levels detected correctly!")
else:
    print(f"\n❌ {len(level_issues)} level detection issues")

In [ ]:
# SAVE RESULTS TO JSON
results_output = {
    "timestamp": datetime.now().isoformat(),
    "total_tests": len(test_results),
    "passed": passed,
    "failed": failed,
    "errors": errors,
    "total_duration_seconds": total_duration,
    "results": [
        {
            "scenario_id": r.scenario_id,
            "scenario_name": r.scenario_name,
            "status": r.status,
            "triple_count": r.triple_count,
            "schema_level": r.schema_level,
            "shacl_shape_count": r.shacl_shape_count,
            "duration_seconds": r.duration_seconds,
            "errors": r.errors,
            "details": r.details
        }
        for r in test_results
    ]
}

output_path = "/lakehouse/default/Files/config/test_results.json"
with open(output_path, 'w') as f:
    json.dump(results_output, f, indent=2)

print(f"\nResults saved to: {output_path}")

In [ ]:
# FINAL SUMMARY
print("\n" + "#"*70)
print("# TEST RUN COMPLETE")
print("#"*70)

if failed == 0 and errors == 0:
    print(f"\n✅ ALL {passed} TESTS PASSED!")
else:
    print(f"\n⚠️  {passed} passed, {failed} failed, {errors} errors")
    print("\nReview failures above and check:")
    print("  1. Input files exist at expected paths")
    print("  2. Expected triple counts are reasonable")
    print("  3. Schema detection logic handles edge cases")

print(f"\nDuration: {total_duration:.1f}s")
print(f"Results: {output_path}")